# Problem Set: Mapping and serving a hospital trust's clinical data with Claude
### Chapter 8 — Linking Ontologies to Data

**Topic coverage:** Keet §8.1 (the data–ontology gap, mappings), §8.2 (materialisation / ETL), §8.3 (query rewriting, OBDA); course notebooks 01–03 of this chapter. Tools: the Chapter 8 mapping engine (R2RML-style class and property maps, a materialiser and a SPARQL-to-SQL rewriter), DSPy + GEPA, LangChain agents, value iteration.

**Level:** Graduate (practice-first). Combines conceptual analysis with building, measuring and optimising a real Claude-backed agent.

**Total points:** 100 (Part A 20 / Part B 30 / Part C 28 / Part D 22); 54 pts are auto-graded by the check cells, the rest are marked by hand against the rubric. **Estimated effort:** 10–12 hours.

**API budget:** ≈ \$6–12 estimated on `claude-opus-5` for a full run (≈ 170 DSPy calls plus 8 agent episodes; GEPA and the C3 noise runs are the largest lines). Nothing in this estimate was measured — record your own figures in the cost cells. Re-running unchanged cells is free: DSPy caches identical requests. Every model call in this notebook is a billed request to Anthropic; the cost lines the problems ask for are part of the answer.

## The brief

Northgate Regional Health Trust is putting an ontology in front of its clinical
database (wards, beds, staff, patients, admissions, transfers, diagnoses, drugs,
prescriptions, lab results — 11 tables). Teams across the Trust file **integration
requests**: *"expose `transfer.from_ward` as `med:fromWard` for the patient-flow
alerting service"*. Every request needs two decisions, and the data-platform lead
wants both made the same way every time:

* **the mapping shape** — does the column become an **IRI** (and of which class) or
  a **literal**? Get it wrong and nothing fails: the graph is silently disconnected,
  and the bed board reports an empty cardiology ward;
* **the execution strategy** — **materialise** the mapped graph (and refresh it after
  updates) or **rewrite** each query into SQL against the live source? Get it wrong
  and the service is either needlessly slow, or it serves stale answers.

You are the ontology engineer on the platform team. Your brief:

1. **Make the silent failure loud.** Measure what a wrong mapping does, and build
   the two detectors the CI will run: a schema-aware lint and a two-path agreement
   test.
2. **Build the grader, then a Claude mapping designer.** Decisions are graded by
   *executing* them — the proposed mapping's triples against the reviewed mapping's,
   the proposed strategy against the MDP's optimal policy — never by string match.
   Improve the designer with GEPA, and report on a **held-out test split** with the
   cost and the run-to-run noise.
3. **Ship a design-review agent** that inspects the schema, *tries* its mapping, prices
   the strategy with the MDP, and only then answers.
4. **Price staleness.** Derive when materialising pays, and show what an agent does
   when staleness is left out of its reward.

The Trust schema and extract, the class maps, the reviewed production mappings and
the 24 integration requests (split 8/8/8 by item, two of each *shape × strategy*
combination per split) are provided in `ch08_agentic.py`. Every gold label is
recomputed from the schema's foreign keys and the MDP when the notebook starts.

## Submission requirements

1. **This notebook, executed top to bottom**, with every `TODO` implemented and every *Your answer* cell written.
2. **The artefacts it saves** under `course/artifacts/` (optimised instructions, run logs) — they are evidence, not by-products.
3. **Honest reporting.** A number without its split, its sample size and its cost earns no credit. Report the runs you did, including the ones that did not improve anything.

The check cells are the marking scheme for the auto-graded points; do not edit them. The final self-test prints your auto-graded subtotal.

> Solutions: `04_solutions.ipynb` — do not open it until you have attempted every problem.

In [ ]:
import sys, json, logging
from pathlib import Path

# Make the course root and this chapter importable no matter where Jupyter started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break
sys.path.insert(0, str(here))

import oe_course
from oe_course import config
from oe_course.grading import Grader

print(json.dumps(oe_course.describe_environment(), indent=1))
config.require_credentials()          # this problem set calls the live Anthropic API
logging.getLogger("dspy").setLevel(logging.WARNING)
GRADER = Grader("Chapter 8 — Linking Ontologies to Data — Mapping and serving a hospital trust's clinical data with Claude")

import dataclasses, time
import dspy
import numpy as np
import pandas as pd
import ch08_toolkit as ch8
import ch08_agentic as A
from oe_course import evaluation as ev, llm, mdp, optimize as opt, agents

print(A.validate_gold())                     # every gold label recomputed from schema + MDP
train, dev, test = A.build_dataset("train"), A.build_dataset("dev"), A.build_dataset("test")
EXAMPLES = {ex.id: ex for ex in train + dev + test}
conn = A.build_trust_database()
print(f"train {len(train)} / dev {len(dev)} / test {len(test)}")
pd.crosstab([pd.Series([e.split for e in EXAMPLES.values()], name="split")],
            [pd.Series([e.gold_object_kind for e in EXAMPLES.values()], name="shape"),
             pd.Series([e.gold_strategy for e in EXAMPLES.values()], name="strategy")])

One dataset row, as the designer will see it (`request`, `workload`, `schema` are the
inputs; the `gold_*` fields are the labels):

In [ ]:
ex = EXAMPLES["fromward-flow"]
print(ex.request, "\n")
print(ex.workload, "\n")
print({k: ex[k] for k in ["gold_object_kind", "gold_target_class", "gold_strategy",
                          "reads_per_update", "refresh_queries"]})

---

## Part A — The silent failure, and the detectors that make it loud

`A.TRUST_MAPPINGS` is the reviewed production mapping: ten class maps and one property
map per request. Its foreign-key properties are IRIs built from the *referenced* table's
class template; everything else is a literal.

In [ ]:
TEMPLATE_CLASS = {c.template: name for name, c in A.CLASS_MAPS.items()}
RANGE = {ex.predicate: ex.gold_target_class for ex in EXAMPLES.values()}  # the ontology's rdfs:range ('none' = datatype property)
FK_MAPS = [p for p in A.TRUST_MAPPINGS["properties"] if p.object_template is not None]
full_graph = ch8.materialise(conn, A.TRUST_MAPPINGS)
print(len(full_graph), "triples;", len(FK_MAPS), "object properties,",
      len(A.TRUST_MAPPINGS["properties"]) - len(FK_MAPS), "datatype properties")
pd.DataFrame(ch8.mapping_report(A.TRUST_MAPPINGS)).tail(8)

### A1. Measure the damage a literal does (10 pts)

Write `probe_query(pmap, target_class) -> ch8.ConjunctiveQuery`: a query that *uses* the
property. For an object property (`target_class` a key of `A.CLASS_MAPS`) it selects
`?s` such that `?s <predicate> ?o` **and** `?o` is an instance of the target class; for a
datatype property (`target_class == "none"`) it selects `?s` and `?o` from the one
property atom.

Then, for **every** foreign-key property map in `FK_MAPS`, break it — the same map with
`object_template=None` (`dataclasses.replace`) swapped into an otherwise unchanged copy
of `A.TRUST_MAPPINGS` — and build `a1` with one row per property and columns
`property` (local name), `triples_correct`, `triples_broken` (size of the full
materialised graph), `materialised_correct`, `materialised_broken`, `rewritten_broken`
(answer counts of the probe query, target = the property's range), and `agree_broken`
(do materialisation and rewriting give identical answers under the broken mapping?).

In writing: why does nothing — the mapper, the triple store, SPARQL, SQL — raise an
error? Where in the Trust would this bug first be *noticed*, and by whom?

In [ ]:
def probe_query(pmap, target_class: str) -> ch8.ConjunctiveQuery:
    # TODO
    raise NotImplementedError

def swap(mappings, new_pmap):
    """A copy of `mappings` with the property map of the same predicate replaced."""
    # TODO
    raise NotImplementedError

a1 = None   # TODO: one row per foreign-key property map

In [ ]:
# Marking scheme for A1 -- do not edit.
with GRADER.check('A1', points=6):
    assert isinstance(a1, pd.DataFrame) and len(a1) == len(FK_MAPS) == 12
    assert set(a1["property"]) == {p.predicate.split("#")[-1] for p in FK_MAPS}
    assert (a1.triples_broken == a1.triples_correct).all(), "the broken graph has as many triples"
    assert (a1.materialised_correct > 0).all() and (a1.materialised_broken == 0).all()
    assert (a1.rewritten_broken == a1.materialised_correct).all()
    assert (~a1.agree_broken.astype(bool)).all()
    # The probe must be a faithful query: on the CORRECT mapping both paths agree, for all 24 properties.
    for p in A.TRUST_MAPPINGS["properties"]:
        q = probe_query(p, RANGE[p.predicate.split("#")[-1]])
        left = ch8.answers_via_materialisation(conn, q, graph=full_graph)
        assert left and left == ch8.answers_via_rewriting(conn, q, A.TRUST_MAPPINGS), p.predicate

**Your answer:**

*(Write your answer here.)*

### A2. Two detectors, and what each one misses (10 pts)

(a) Write `lint_mapping(conn, mappings) -> list[dict]`, a schema-aware lint returning one
`{"property": <local name>, "issue": ...}` per defective property map, where `issue` is

* `"fk-as-literal"` — the object column is a **declared** foreign key (`A.foreign_keys`)
  but the object is a literal;
* `"wrong-target"` — a declared foreign key whose IRI template is not the template of
  the referenced table's class (`A.class_for_table`, `A.CLASS_MAPS`);
* `"attribute-as-iri"` — not a foreign key, but mapped to an IRI.

(b) Three realistic defects are provided below. For each, build `a2` with columns
`defect`, `lint_issues` (the issues your lint reports), `agreement_detects` (do the two
execution paths disagree on your A1 probe query, written against the property's
**ontology range** `RANGE`?) and `connected_share` (share of the property's objects that
are typed instances of its range, from `A.object_profile`; 0 for a datatype property).

In writing: which detector catches which defect, and why? Name one defect *neither*
catches, and say what would. Which of the two would you make a blocking CI check?

In [ ]:
by_name = {p.predicate.split("#")[-1]: p for p in A.TRUST_MAPPINGS["properties"]}
DEFECTS = {
    # a foreign key mapped as a literal (A1's bug)
    "fk-as-literal": dataclasses.replace(by_name["inWard"], object_template=None),
    # a plain attribute given an IRI template: 'cardiology' becomes a resource
    "attribute-as-iri": dataclasses.replace(by_name["speciality"],
                                            object_template=A.CLASS_MAPS["Ward"].template),
    # the right idea, the wrong table: ward ids turned into Bed IRIs (bed 1..3 exist!)
    "wrong-target": dataclasses.replace(by_name["inWard"],
                                        object_template=A.CLASS_MAPS["Bed"].template),
}

In [ ]:
def lint_mapping(conn, mappings) -> list[dict]:
    # TODO (a)
    raise NotImplementedError

a2 = None   # TODO (b): one row per defect in DEFECTS

In [ ]:
# Marking scheme for A2 -- do not edit.
with GRADER.check('A2', points=5):
    assert lint_mapping(conn, A.TRUST_MAPPINGS) == [], "the reviewed mapping must lint clean"
    # Foreign keys without an _id suffix, and TEXT foreign keys, must be caught too.
    for name in ["fromWard", "prescribedBy", "hasDisorder", "prescribes"]:
        bad = swap(A.TRUST_MAPPINGS, dataclasses.replace(by_name[name], object_template=None))
        assert lint_mapping(conn, bad) == [{"property": name, "issue": "fk-as-literal"}], name
    assert isinstance(a2, pd.DataFrame) and set(a2["defect"]) == set(DEFECTS)
    got = a2.set_index("defect")
    for defect in DEFECTS:
        assert list(got.loc[defect, "lint_issues"]) == [defect], defect
    assert bool(got.loc["fk-as-literal", "agreement_detects"])
    assert bool(got.loc["wrong-target", "agreement_detects"])
    assert not bool(got.loc["attribute-as-iri", "agreement_detects"])
    assert (got["connected_share"] == 0).all()

**Your answer:**

*(Write your answer here.)*

---

## Part B — Build the grader, then the Claude mapping designer

The guidelines a scorer can report are in `A.OBDA_RULEBOOK` — ids plus the sentence
GEPA's reflection step will read:

In [ ]:
for rule in A.OBDA_RULEBOOK:
    print(f"{rule.id:34s} {rule.description[:84]}")

### B1. A grader that executes the decision (14 pts)

Implement `obda_scorer(gold, pred) -> ev.ScoreReport`. `pred` has string fields
`object_kind`, `target_class`, `strategy`. Normalise each (strip whitespace, quotes and
a trailing full stop; lower-case; accept the US spelling `materialize`; match class
names case-insensitively). Then:

| stage | outcome | score | `violated` |
|---|---|---|---|
| contract | `object_kind` ∉ {iri, literal} or `strategy` ∉ {materialise, rewrite} | 0.0 overall | `["answer-in-the-contract"]` |
| mapping (0.5) | `iri` with a `target_class` that is not a key of `A.CLASS_MAPS` | +0 | `answer-in-the-contract` |
| | the predicted property map produces **exactly** the gold map's triples (`A.property_triples`) | +0.5 | — |
| | gold is an IRI, predicted a literal | +0 | `iri-for-foreign-keys` |
| | gold is a literal, predicted an IRI | +0 | `literal-for-attributes` |
| | both IRIs, wrong class | +0 | `reference-the-right-table` |
| strategy (0.5) | equals `gold.gold_strategy` | +0.5 | — |
| | gold materialise, got rewrite / gold rewrite, got materialise | +0 | `materialise-when-refresh-pays` / `rewrite-when-refresh-does-not-pay` |

Mapping violations come before strategy violations. Build the predicted property map
from the gold one (`A.gold_property_map(A.REQUEST_BY_ID[gold.id], conn)`) with only the
object template changed; for a literal the target class is ignored. The `notes` are what
GEPA reads — make them *evidence*, not verdicts: for a foreign key mapped to a literal,
say how many links now join to nothing (use the word **disconnected**); for a wrong
class, where the objects landed (`A.object_profile`); for a strategy error, the
**break-even** number of reads per update against the workload's actual number.

In [ ]:
SCORER_CONN = A.build_trust_database()
MAPPING_RULES = {"answer-in-the-contract", "iri-for-foreign-keys",
                 "reference-the-right-table", "literal-for-attributes"}

def _norm(value) -> str:
    return str(value or "").strip().strip("'\"`").rstrip(".").strip().lower()

def obda_scorer(gold, pred) -> ev.ScoreReport:
    kind = _norm(getattr(pred, "object_kind", ""))
    strategy = _norm(getattr(pred, "strategy", ""))
    target = _norm(getattr(pred, "target_class", ""))
    gold_map = A.gold_property_map(A.REQUEST_BY_ID[gold.id], SCORER_CONN)
    # TODO: contract stage, mapping half, strategy half (see the table)
    raise NotImplementedError

In [ ]:
# Marking scheme for B1 -- do not edit.
with GRADER.check('B1', points=12):
    P = lambda k, t, s: dspy.Prediction(object_kind=k, target_class=t, strategy=s)
    fk = EXAMPLES["fromward-flow"]          # transfer.from_ward -> Ward; rewrite
    attr = EXAMPLES["nhsnumber-mpi"]        # patient.nhs_number -> literal; materialise
    cases = [
        (fk, P("iri", "Ward", "rewrite"), 1.0, []),
        (fk, P(" IRI.", "ward", "Rewrite"), 1.0, []),
        (fk, P("iri", "Ward", "materialise"), 0.5, ["rewrite-when-refresh-does-not-pay"]),
        (fk, P("literal", "none", "rewrite"), 0.5, ["iri-for-foreign-keys"]),
        (fk, P("iri", "Transfer", "rewrite"), 0.5, ["reference-the-right-table"]),
        (fk, P("literal", "none", "materialise"), 0.0,
         ["iri-for-foreign-keys", "rewrite-when-refresh-does-not-pay"]),
        (fk, P("iri", "Hospital", "rewrite"), 0.5, ["answer-in-the-contract"]),
        (fk, P("a foreign key", "Ward", "rewrite"), 0.0, ["answer-in-the-contract"]),
        (fk, P("iri", "Ward", ""), 0.0, ["answer-in-the-contract"]),
        (attr, P("literal", "none", "materialise"), 1.0, []),
        (attr, P("literal", "Patient", "materialize"), 1.0, []),
        (attr, P("iri", "Patient", "materialise"), 0.5, ["literal-for-attributes"]),
        (attr, P("literal", "none", "rewrite"), 0.5, ["materialise-when-refresh-pays"]),
        (EXAMPLES["hasdisorder-cohort"], P("literal", "none", "materialise"), 0.5,
         ["iri-for-foreign-keys"]),
    ]
    for gold, pred, score, violated in cases:
        r = obda_scorer(gold, pred)
        assert abs(r.score - score) < 1e-9 and r.violated == violated, (gold.id, pred, r.score, r.violated)
        assert all(v in A.OBDA_RULEBOOK for v in r.violated)
    r = obda_scorer(fk, P("literal", "none", "materialise"))
    assert any("disconnected" in n for n in r.notes), "say that the graph is disconnected"
    assert any("break-even" in n for n in r.notes), "give the break-even number of reads"
    # Gold answers score 1.0 on every item.
    for ex in EXAMPLES.values():
        assert obda_scorer(ex, P(ex.gold_object_kind, ex.gold_target_class, ex.gold_strategy)).score == 1.0

### B2. The mapping designer as a DSPy program (7 pts)

Write a signature `MappingDecision` with inputs `request`, `workload` and `schema`, and
outputs `object_kind`, `target_class`, `strategy` and `justification`; and a factory
`OBDADesigner(instruction)` returning a `dspy.Module` with a single `dspy.Predict` whose
instruction is `instruction`. The field descriptions are part of the prompt — state the
allowed labels in them (manual marks). Configure Claude and run the program once on
`train[0]`.

In [ ]:
BASELINE_INSTRUCTION = A.BASELINE_INSTRUCTION

# TODO: class MappingDecision(dspy.Signature): ...
# TODO: def OBDADesigner(instruction=BASELINE_INSTRUCTION): ...

lm = llm.configure_dspy()
smoke = None     # TODO: OBDADesigner()(**train[0].inputs())

raise NotImplementedError

In [ ]:
# Marking scheme for B2 -- do not edit.
with GRADER.check('B2', points=4):
    assert set(MappingDecision.input_fields) == {"request", "workload", "schema"}
    assert {"object_kind", "target_class", "strategy"} <= set(MappingDecision.output_fields)
    program = OBDADesigner("custom instruction")
    assert len(list(program.named_predictors())) == 1
    assert opt.instruction_of(program) == "custom instruction"
    assert all(isinstance(getattr(smoke, f), str) for f in ["object_kind", "target_class", "strategy"])

### B3. Baseline on dev — which decision is it getting wrong? (9 pts)

The score mixes two independent decisions, so a mean alone cannot tell you what to fix.
Write `dimension_accuracy(result) -> {"mapping": float, "strategy": float}` from the
per-row `violated` lists of an `ev.evaluate_dataset` result (a row's mapping is right
iff it violates none of `MAPPING_RULES`; its strategy is right iff it violates neither
strategy rule nor `answer-in-the-contract`).

Evaluate the baseline on `dev` inside `llm.meter(lm)`; store `baseline_dev`,
`baseline_dev_cost` and `baseline_dims`. Then write an **error analysis**: for each
non-perfect item, which guideline, and *why* you think Claude chose as it did. Did it
follow the rule of thumb "live data → rewrite" where the numbers said otherwise?

In [ ]:
def dimension_accuracy(result) -> dict:
    # TODO
    raise NotImplementedError

# TODO: baseline_dev, baseline_dev_cost, baseline_dims

In [ ]:
# Marking scheme for B3 -- do not edit.
with GRADER.check('B3', points=3):
    fake = {"rows": [{"item": "a", "score": 1.0, "violated": []},
                     {"item": "b", "score": 0.5, "violated": ["iri-for-foreign-keys"]},
                     {"item": "c", "score": 0.5, "violated": ["materialise-when-refresh-pays"]},
                     {"item": "d", "score": 0.0, "violated": ["answer-in-the-contract"]}]}
    assert dimension_accuracy(fake) == {"mapping": 0.5, "strategy": 0.5}
    assert baseline_dev["n"] == len(dev) == 8
    assert {r["item"] for r in baseline_dev["rows"]} == {ex.id for ex in dev}
    assert baseline_dims == dimension_accuracy(baseline_dev)
    assert {"calls", "usd"} <= set(baseline_dev_cost)

**Your answer:**

*(Write your answer here.)*

---

## Part C — Optimise with GEPA — and report it honestly

**Optimise on `train`, select on `dev`, report on `test`** — and put a cost and a noise
estimate next to every number. Every metric call is a billed request.

### C1. A budgeted GEPA run with a held-out report (12 pts)

1. Build the GEPA feedback metric from your scorer and `A.OBDA_RULEBOOK`, and a separate
   reflection LM (`llm.reflection_lm()`).
2. Run `opt.run_gepa` on `train` with `valset=dev` and `max_metric_calls=GEPA_BUDGET`
   inside `llm.meter(lm, reflect)`; store the cost in `gepa_cost`.
3. Compare the baseline and the tuned program on **`test`** with `opt.compare` (store as
   `c1`), print `c1.report()`, and report `dimension_accuracy` before and after.
4. Save the tuned instruction to `config.artifacts_dir() /
   "ch08_obda_designer_instruction.txt"`.

In writing: read the instruction diff. Which guidelines did GEPA write down, which did it
miss — and was each missed guideline ever *violated* on train? Did it write the
break-even rule as a rule, or memorise training workloads?

In [ ]:
GEPA_BUDGET = 60
# TODO: gepa_metric, reflect, tuned (inside llm.meter -> gepa_cost), c1, save the instruction
instruction_path = config.artifacts_dir() / "ch08_obda_designer_instruction.txt"

raise NotImplementedError

In [ ]:
# Marking scheme for C1 -- do not edit.
with GRADER.check('C1', points=6):
    ids = lambda rows: {ex.id for ex in rows}
    assert not (ids(test) & (ids(train) | ids(dev))), "the test split must be untouched"
    assert {r["item"] for r in c1.after["rows"]} == ids(test), "report on the test split"
    assert c1.before["n"] == c1.after["n"] == len(test)
    assert instruction_path.is_file()
    assert instruction_path.read_text(encoding="utf-8") == opt.instruction_of(tuned)
    assert {"calls", "usd"} <= set(gepa_cost) and GEPA_BUDGET <= 200

**Your answer:**

*(Write your answer here.)*

### C2. Three programs, two decisions, one table (8 pts)

Add the **hand-written guidelines** —
`A.OBDA_RULEBOOK.as_guidelines(BASELINE_INSTRUCTION)` — as a third contender. Evaluate
baseline, guidelines and GEPA-tuned on `test`, each inside its own `llm.meter`, and build
`c2` with columns `program` (`"baseline"`, `"guidelines"`, `"gepa"`), `test_mean`,
`mapping_acc`, `strategy_acc`, `violations`, `eval_usd`, `optimisation_usd` (the GEPA
cost for `"gepa"`, 0 otherwise) and `instruction_chars`.

In writing: which would you deploy for the platform team, and what would change your
mind?

In [ ]:
guidelines_instruction = A.OBDA_RULEBOOK.as_guidelines(BASELINE_INSTRUCTION)
c2 = None    # TODO

raise NotImplementedError

In [ ]:
# Marking scheme for C2 -- do not edit.
with GRADER.check('C2', points=3):
    assert isinstance(c2, pd.DataFrame)
    assert set(c2["program"]) == {"baseline", "guidelines", "gepa"}
    assert {"test_mean", "mapping_acc", "strategy_acc", "violations", "eval_usd",
            "optimisation_usd", "instruction_chars"} <= set(c2.columns)
    assert c2.set_index("program").loc[["baseline", "guidelines"], "optimisation_usd"].eq(0).all()
    assert c2[["mapping_acc", "strategy_acc"]].apply(lambda s: s.between(0, 1)).all().all()

**Your answer:**

*(Write your answer here.)*

### C3. Is the difference bigger than the noise? (8 pts)

Eight test items at temperature 1.0: one item is 0.0625 of mean score per half-decision.
With caching **disabled** (`fresh = llm.dspy_lm(cache=False)` and
`with dspy.context(lm=fresh): ...`), run the baseline and the GEPA-tuned programs on
`test` three times each; store the mean scores in `runs = {"baseline": [..3..], "gepa":
[..3..]}` and the spend in `c3_cost`.

Set `verdict_c3` to `"significant"` if `|mean_gepa - mean_baseline| > 2 * max(sd_gepa,
sd_baseline)` (sample standard deviations), else `"not significant"`. In writing: what
does this say about C1, and what would a credible evaluation of this system need?

In [ ]:
fresh = llm.dspy_lm(cache=False)
runs = {"baseline": [], "gepa": []}
# TODO: fill runs (inside llm.meter(fresh) as c3_cost), then compute verdict_c3
verdict_c3 = None

raise NotImplementedError

In [ ]:
# Marking scheme for C3 -- do not edit.
with GRADER.check('C3', points=4):
    import statistics as _st
    assert set(runs) == {"baseline", "gepa"} and all(len(v) == 3 for v in runs.values())
    assert all(0.0 <= x <= 1.0 for v in runs.values() for x in v)
    assert {"calls", "usd"} <= set(c3_cost)
    _m = {k: _st.mean(v) for k, v in runs.items()}
    _s = {k: _st.stdev(v) for k, v in runs.items()}
    _expected = ("significant" if abs(_m["gepa"] - _m["baseline"]) > 2 * max(_s.values())
                 else "not significant")
    assert verdict_c3 == _expected

**Your answer:**

*(Write your answer here.)*

---

## Part D — A design-review agent, and the price of staleness

The agent gets four tools over the Trust database: `inspect_schema` (tables, columns,
declared foreign keys), `list_classes` (class maps and IRI templates), `check_mapping`
(materialise one candidate property map and report where its objects land: typed
instances, literals, dangling IRIs) and `price_strategy` (solve the materialise-or-
rewrite MDP for a workload's numbers). A design that has not been *tried* is a guess.

In [ ]:
ws_demo = A.TrustWorkspace()
demo_tools = {t.name: t for t in A.build_trust_tools(ws_demo)}
print(demo_tools["check_mapping"].invoke({"table": "transfer", "subject_column": "id",
      "object_column": "from_ward", "object_kind": "literal"}))
print(demo_tools["check_mapping"].invoke({"table": "transfer", "subject_column": "id",
      "object_column": "from_ward", "object_kind": "iri", "target_class": "Ward"}))
print(demo_tools["price_strategy"].invoke({"reads_per_update": 1, "refresh_cost_in_queries": 5}))
print(ws_demo.log.names())

### D1. Build and evaluate the design-review agent (12 pts)

1. Write `REVIEW_PROMPT`, the agent's system prompt. It must make the agent inspect the
   schema, **try its mapping with `check_mapping`** and accept it only if every object
   lands on the intended class (or is a literal, for an attribute), extract the
   workload's numbers and **price the strategy with `price_strategy`**, and end with a
   JSON object `{"object_kind": ..., "target_class": ..., "strategy": ..., "verified":
   true|false}`.
2. Write `parse_decision(text) -> dict | None`: the last JSON object in the text (code
   fences and surrounding prose allowed) that has `object_kind` and `strategy`, returned
   as `{"object_kind", "target_class", "strategy"}` strings (`target_class` defaults to
   `"none"`); `None` if there is none.
3. For every **test** request, build a fresh `A.TrustWorkspace()`, its tools, an agent
   (`agents.build_agent(ws, system_prompt=..., tools=...)`), and run it on
   `ex.request + "\n\nWorkload: " + ex.workload`. Collect `d1` with columns `id`,
   `object_kind`, `target_class`, `strategy` (parsed, `None` when unparseable), `score`
   (your `obda_scorer`; 0 when unparseable), `checked_mapping`, `priced` (did the log
   contain those tool calls?), `tool_calls`, `usd_estimate` (`llm.chat_usage`).

In writing: report the mean score with its n and the cost per request; compare with the
DSPy designer on the same test items; examine every error. Why is `check_mapping` worth
more than a sentence in the prompt?

In [ ]:
REVIEW_PROMPT = """TODO"""

def parse_decision(text: str) -> dict | None:
    # TODO
    raise NotImplementedError

d1 = None   # TODO: run the agent on every test request

In [ ]:
# Marking scheme for D1 -- do not edit.
with GRADER.check('D1', points=5):
    assert parse_decision('{"object_kind": "iri", "target_class": "Ward", "strategy": "rewrite", "verified": true}') \
        == {"object_kind": "iri", "target_class": "Ward", "strategy": "rewrite"}
    assert parse_decision('Checked.\n```json\n{"object_kind": "literal", "strategy": "materialise"}\n```') \
        == {"object_kind": "literal", "target_class": "none", "strategy": "materialise"}
    assert parse_decision('{"triples": 6} then {"object_kind": "iri", "target_class": "Staff", '
                          '"strategy": "rewrite"}')["target_class"] == "Staff"
    assert parse_decision('I would map it as an IRI.') is None
    assert parse_decision('{"verdict": "yes"}') is None
    assert isinstance(d1, pd.DataFrame) and list(d1["id"]) == [ex.id for ex in test]
    assert {"id", "object_kind", "target_class", "strategy", "score", "checked_mapping", "priced",
            "tool_calls", "usd_estimate"} <= set(d1.columns)
    for ex, row in zip(test, d1.to_dict("records")):
        if row["object_kind"] is None:
            assert row["score"] == 0.0
        else:
            pred = dspy.Prediction(object_kind=row["object_kind"], target_class=row["target_class"],
                                   strategy=row["strategy"])
            assert abs(row["score"] - obda_scorer(ex, pred).score) < 1e-9, ex.id

**Your answer:**

*(Write your answer here.)*

### D2. When does materialising pay — and what if staleness were free? (10 pts)

`A.MaterialisationMDP.for_workload(r, N)` builds a periodic workload — `r` queries, then
one update, twice — with a refresh priced at `N` rewritten queries (serving costs:
rewrite 0.30, from the copy 0.05; a correct answer earns 1; a stale answer earns 0).

1. Derive, on paper, the break-even reads per update `r*` above which refreshing and
   serving from the copy beats rewriting, as a function of `N` and the two serving costs;
   implement it as `breakeven_reads(N, cost_rewrite=0.30, cost_materialised=0.05)`.
2. Verify it: for `N` in `[1, 3, 6]` and `r` in `1..12`, solve the MDP with
   `mdp.value_iteration`, roll out the greedy policy, and record in `d2` (columns
   `refresh_queries`, `reads_per_update`, `share_from_copy`, `materialises`) whether
   the optimal policy serves most queries from the copy.
3. Write `LatencyOnlyMDP`, a subclass whose reward ignores staleness (a stale copy's
   answer earns `1 - cost_materialised` like a fresh one). On the default workload
   `"qquqqquq"`, count the queries the optimal policy answers from a **stale** copy under
   each reward: `stale_latency_only` and `stale_priced`.

In writing: interpret `r*` for the Trust's requests, and name two things the MDP leaves
out that a platform team would have to add before trusting it for a real SLA.

In [ ]:
M = A.MaterialisationMDP()            # workload 'qquqqquq', refresh 0.50
V, pi = mdp.value_iteration(M)
state = M.initial_state()
while not M.is_terminal(state):
    print("  " + M.describe(state, pi[state]))
    state = M.transition(state, pi[state])[0][1]
print(f"V*(s0) = {V[M.initial_state()]:.3f}")

In [ ]:
def breakeven_reads(N: float, cost_rewrite: float = 0.30,
                    cost_materialised: float = 0.05) -> float:
    # TODO (1)
    raise NotImplementedError

d2 = None                     # TODO (2)

class LatencyOnlyMDP(A.MaterialisationMDP):
    pass                      # TODO (3)

stale_latency_only = None     # TODO (3)
stale_priced = None

In [ ]:
# Marking scheme for D2 -- do not edit.
with GRADER.check('D2', points=6):
    assert abs(breakeven_reads(2) - 2.4) < 1e-9 and abs(breakeven_reads(4, 0.5, 0.1) - 5.0) < 1e-9
    assert isinstance(d2, pd.DataFrame) and set(d2["refresh_queries"]) == {1, 3, 6}
    assert set(d2["reads_per_update"]) == set(range(1, 13))
    for _, row in d2.iterrows():
        r_star = breakeven_reads(row.refresh_queries)
        if abs(row.reads_per_update - r_star) > 0.25:
            assert bool(row.materialises) == (row.reads_per_update > r_star), dict(row)
    assert issubclass(LatencyOnlyMDP, A.MaterialisationMDP)
    _s = A.ServeState(0, False)
    assert abs(LatencyOnlyMDP("q").transition(_s, "serve:materialised")[0][2] - 0.95) < 1e-9
    assert stale_priced == 0 and stale_latency_only == 6

**Your answer:**

*(Write your answer here.)*

---

## Grading rubric

| Part | Problem | Auto | Manual | Pts |
|---|---|---|---|---|
| A | A1 Measure the damage a literal does | 6 | 4 | 10 |
| A | A2 Two detectors, and what each one misses | 5 | 5 | 10 |
| B | B1 A grader that executes the decision | 12 | 2 | 14 |
| B | B2 The mapping designer as a DSPy program | 4 | 3 | 7 |
| B | B3 Baseline on dev — which decision is it getting wrong? | 3 | 6 | 9 |
| C | C1 A budgeted GEPA run with a held-out report | 6 | 6 | 12 |
| C | C2 Three programs, two decisions, one table | 3 | 5 | 8 |
| C | C3 Is the difference bigger than the noise? | 4 | 4 | 8 |
| D | D1 Build and evaluate the design-review agent | 5 | 7 | 12 |
| D | D2 When does materialising pay — and what if staleness were free? | 6 | 4 | 10 |
| **Total** | | | | **100** |

*Manual* points are for the written answers and for the quality of the engineering evidence (splits, sample sizes, cost, honest negative results).

---

## Self-test

Re-run this cell after finishing. It lists every auto-graded problem and whether its check passed.

In [ ]:
AUTO_GRADED = ['A1', 'A2', 'B1', 'B2', 'B3', 'C1', 'C2', 'C3', 'D1', 'D2']
_ = GRADER.summary(AUTO_GRADED)